In [1]:
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
from pprint import pprint
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
match_df = pl.read_parquet('Skillcorner data/RealMadrid/matches.parquet')
# Data is only available for "closed" matches
match_df = match_df.filter(pl.col('status') == 'closed')
match_ids = match_df['id'].to_list()
print('Num of matches:', len(match_ids))

Num of matches: 98


In [3]:
REAL_MADRID_TEAM_ID = 262

PITCH_X_M = 105  # meters
PITCH_Y_M = 68  # meters
GOAL_WIDTH_M = 7.32
GOAL_Y_TOP = 0 + (GOAL_WIDTH_M/2)
GOAL_Y_BOT = 0 - (GOAL_WIDTH_M/2)

dynam_dfs = []
for match_id in match_ids:
  try:
    df = pl.read_parquet(f'Skillcorner data/RealMadrid/dynamic/{match_id}.parquet')
    df = df.with_columns(
      pl.col('trajectory_direction_id').cast(pl.Float64)
    ) 
    dynam_dfs.append(df)
  except:
    continue

dynam_df = pl.concat(dynam_dfs)

Calculate distance and angles

In [ ]:
shots_df = dynam_df.filter(
  pl.col('end_type') == 'shot'
)

shots_df = shots_df.with_columns(
  (((PITCH_X_M/2 - pl.col('x_end')).pow(2) + ((PITCH_Y_M/2) - pl.col('y_end')).pow(2)).sqrt()).alias('distance_to_goal')
)

shots_x_np = shots_df['x_end'].to_numpy()
shots_y_np = shots_df['y_end'].to_numpy()

vec0 = np.vstack([PITCH_X_M/2 - shots_x_np, GOAL_Y_TOP - shots_y_np])
vec1 = np.vstack([PITCH_X_M/2 - shots_x_np, GOAL_Y_BOT - shots_y_np])

dotp = np.sum(vec0 * vec1, axis=0)
norm0 = np.linalg.norm(vec0, axis=0)
norm1 = np.linalg.norm(vec1, axis=0)
denom = norm0 * norm1

cos_ang = np.zeros_like(dotp, dtype=np.float64)
valid = denom > 0
cos_ang[valid] = dotp[valid] / denom[valid]
cos_ang = np.clip(cos_ang, -1.0, 1.0)

angle_rad = np.arccos(cos_ang)
angle_deg = np.degrees(angle_rad)

angle_deg[~valid] = np.nan

shots_df = shots_df.with_columns(
  pl.Series('angle_to_goal_rad', angle_rad),
  pl.Series('angle_to_goal_deg', angle_deg)
)

In [5]:
xg_model = smf.glm(
  formula='lead_to_goal ~ distance_to_goal + angle_to_goal_rad',
  data=shots_df.select('lead_to_goal', 'distance_to_goal', 'angle_to_goal_rad').to_pandas(),
  family=sm.families.Binomial(),
).fit()

print(xg_model.summary())

                               Generalized Linear Model Regression Results                               
Dep. Variable:     ['lead_to_goal[False]', 'lead_to_goal[True]']   No. Observations:                 2360
Model:                                                       GLM   Df Residuals:                     2357
Model Family:                                           Binomial   Df Model:                            2
Link Function:                                             Logit   Scale:                          1.0000
Method:                                                     IRLS   Log-Likelihood:                -810.13
Date:                                           Mon, 17 Aug 2026   Deviance:                       1620.3
Time:                                                   22:01:21   Pearson chi2:                 2.31e+03
No. Iterations:                                                5   Pseudo R-squ. (CS):            0.06479
Covariance Type:                              

In [6]:
intercept, dist_coef, ang_coef = xg_model.params.to_list()

xG = 1 / (1 + np.exp(intercept + dist_coef * shots_df['distance_to_goal'] + ang_coef * shots_df['angle_to_goal_rad']))

shots_df = shots_df.with_columns(pl.Series('xG', xG))
rma_xg = shots_df.filter(pl.col('team_id') == REAL_MADRID_TEAM_ID).group_by('match_id').agg(pl.col('xG').sum()).sort('match_id')
oppo_xg = shots_df.filter(pl.col('team_id') != REAL_MADRID_TEAM_ID).group_by('match_id').agg(pl.col('xG').sum()).sort('match_id')

In [ ]:
rma_xg.write_parquet('rma_xg.parquet')
oppo_xg.write_parquet('oppo_xg.parquet')